# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. The dataset covers demographic details and mental health indicators of residents in Kilifi County, Kenya, with unique AI-ready data standards.

### Dataset Source
Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

In [ ]:
# Ensure mlcroissant is installed (run only if necessary)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata as an object and display key information
print(f"Dataset title: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Dataset version: {dataset.metadata.version}")
print(f"Date published: {dataset.metadata.datePublished}")
print(f"Dataset license: {dataset.metadata.license}")

## 2. Data Overview
List available record sets, fields, and their `@id` values for referencing throughout this notebook.

In [ ]:
# Accessing record sets by their @id
record_sets = dataset.record_sets
print("Available record sets and field structure:")

for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    # List fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - Field @id: {field['@id']}, Name: {field.get('name', 'N/A')}")
    # List columns
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print("  Columns:")
    for column in columns:
        print(f"    - Column @id: {column['@id']}, Name: {column.get('name', 'N/A')}")
    print()

## 3. Data Extraction
Load data from all record sets into pandas DataFrames. Record set and field `@id`s are used for referencing.

In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Display columns and a preview of the main record set
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"Columns for main record set @{main_rs_id}:\n")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping on numeric and categorical fields.

Use the `@id`s from the overview section for referencing.

In [ ]:
# Choose the record set and numeric field for demonstration
record_set_id = main_rs_id    # e.g., the survey record set
df = dataframes.get(record_set_id, pd.DataFrame())

if not df.empty:
    # Choose a likely numeric field based on dataset description
    possible_numeric_fields = [col for col in df.columns if 'score' in col.lower() or 'age' in col.lower()]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]

    # Set threshold (e.g., for GAD-7 score)
    threshold = 10
    if numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered rows where {numeric_field} > {threshold}:")
        display(filtered_df.head())
    else:
        print(f"Numeric field '{numeric_field}' not found in DataFrame.")

    # Normalize the numeric field
    if not filtered_df.empty:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered group:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field, e.g., 'village' or similar
        possible_group_fields = [col for col in df.columns if 'village' in col.lower() or 'gender' in col.lower() or 'education' in col.lower()]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
else:
    print("No data available in main record set.")

## 5. Visualization
Visualize distributions and relationships of selected fields.

In [ ]:
# Simple histogram of the numeric field (if available)
if not df.empty and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Boxplot for group field (if available)
if not df.empty and group_field:
    plt.figure(figsize=(10, 5))
    df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"Boxplot of {numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using Croissant schema and `mlcroissant`. We reviewed the record sets, extracted and filtered data, normalized numeric fields, and visualized key distributions. 

Further analysis may focus on deeper relationships between demographic and mental health indicators, and on leveraging dataset standards for AI readiness and reproducible research.